# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

Croissant JSON-LD: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.
Below, we list the available record sets in the dataset with their `@id`s and the fields they contain. This is necessary for referencing in all further operations.

In [ ]:
# List record sets and their fields by their @id
record_sets = list(dataset.record_sets)

if not record_sets:
    print("No explicit record sets found in the schema. Attempting to infer from data files...")
    # Fallback: Try loading available records (default behavior) and get available field names.
    inferred = None
    try:
        sample_records = list(dataset.records())
        if sample_records:
            inferred = sample_records[0]
            print("Default record set (implicit): Fields available:")
            for k in inferred.keys():
                print(f"  - {k}")
    except Exception as e:
        print(f"No available record sets or records: {e}")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        fields = dataset.fields(record_set=rs['@id'])
        for field in fields:
            print(f"  Field @id: {field['@id']}, name: {field.get('name', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All entities are referenced by their `@id`. If only one record set is available and/or implicit, use the default.

In [ ]:
# -- Extract data --
# If there are no named record sets, we use the default (None), else collect by explicit @id
record_set_ids = []
if not dataset.record_sets:
    record_set_ids = [None]  # Implicit default record set
else:
    record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id if rs_id else 'default'] = pd.DataFrame(records)
    print(f"Loaded {len(records)} rows from record set: {rs_id if rs_id else 'default'}")

# Show columns present
main_rs_id = record_set_ids[0] if record_set_ids else 'default'
if main_rs_id is None:
    main_rs_id = 'default'
print("Fields/columns in the main record set:")
print(dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates removing outliers, transforming data distributions, and grouping by key attributes.

In [ ]:
# EDA example: Filter by a numeric field (e.g., 'MW_kDa' for molecular weight in kilodaltons if present)
df = dataframes[main_rs_id]

# Try to find a numeric field. Prioritize typical ones for protein MS data
import numpy as np
possible_numeric_fields = ['MW_kDa', 'Coverage_perc', 'Unique_peptides', 'Peptides', 'PSMs', 'Score', 'pI', 'Abundance']
numeric_field = None
for field in df.columns:
    for candidate in possible_numeric_fields:
        if candidate.lower() in field.lower():
            numeric_field = field
            break
    if numeric_field:
        break
if not numeric_field:
    # fallback to the first float-like field
    for col in df.select_dtypes(include=[np.number]).columns:
        numeric_field = col
        break
assert numeric_field is not None, "No numeric field found in data."

print(f"Using numeric field: {numeric_field}")

# Filter: Only keep records above the mean value
threshold = df[numeric_field].mean()
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} remaining of {len(df)}")

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records (first 5 rows):")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field if available
possible_group_fields = ['Description', 'Gene', 'Sample', 'Type', 'ProteinID', 'Protein Name']
group_field = None
for field in df.columns:
    for candidate in possible_group_fields:
        if candidate.lower() in field.lower():
            group_field = field
            break
    if group_field:
        break

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped mean {numeric_field} by {group_field} (first 5 rows):")
    display(grouped_df.head())
else:
    print("No obvious categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field (e.g., histogram), and the relationship between fields if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

if group_field:
    # If grouping available, boxplot or barplot
    plt.figure(figsize=(10,5))
    order = df[group_field].value_counts().head(10).index
    sns.boxplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(order)], order=order)
    plt.title(f"{numeric_field} by {group_field} (Top 10 group categories)")
    plt.xlabel(group_field)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded the Croissant-encoded dataset, identified available fields via their `@id`s, and demonstrated basic exploratory analysis and visualization steps on mass spectrometry protein data. Further analyses could include feature engineering, advanced statistical testing, or integration with other protein or biological datasets. For all programmatic access to this or related datasets, refer to record sets, fields, and columns using their precise Croissant `@id` as shown above.